In [13]:
import pandas as pd

# reading
votes = pd.read_csv('data/HS119_votes.csv')
members = pd.read_csv('data/HS119_members.csv')
roll_calls = pd.read_csv('data/HS119_rollcalls.csv')

house_votes = votes[votes["chamber"] == "House"]
senate_votes = votes[votes["chamber"] == "Senate"]

house_members = members[members["chamber"] == "House"]
senate_members = members[members["chamber"] == "Senate"]

# mapping
def convert_vote(code):
    if code in [1, 2, 3]:
        return 1;
    elif code in [4, 5, 6]:
        return -1;
    else:
        return 0;

house_votes["votes_numeric"] = votes["cast_code"].apply(convert_vote)
senate_votes["votes_numeric"] = votes["cast_code"].apply(convert_vote)


/var/folders/d2/lph_5phn21vbnn9wstbxvnkm0000gn/T/ipykernel_11826/4198546310.py:23: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/d2/lph_5phn21vbnn9wstbxvnkm0000gn/T/ipykernel_11826/4198546310.py:24: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [18]:
house_matrix = house_votes.pivot_table(index = "icpsr", columns = "rollnumber", values = "votes_numeric", fill_value = 0)
senate_matrix = senate_votes.pivot_table(index = "icpsr", columns = "rollnumber", values = "votes_numeric", fill_value = 0)
from numpy import matrix
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
coords_house = pca.fit_transform(house_matrix)
coords_senate = pca.fit_transform(senate_matrix)

In [21]:
import plotly.express as px
df_house = pd.DataFrame(coords_house, columns=["PC1", "PC2"])
df_house["icpsr"] = house_matrix.index

df_house = df_house.merge(
    house_members[["icpsr", "party_code", "bioname"]],
    on="icpsr"
)


fig = px.scatter(
    df_house,
    title="House",
    x="PC1",
    y="PC2",
    color="party_code",
    hover_name="bioname"
)

fig.show()

In [22]:
import plotly.express as px
df_senate = pd.DataFrame(coords_senate, columns=["PC1", "PC2"])
df_senate["icpsr"] = senate_matrix.index

df_senate = df_senate.merge(
    senate_members[["icpsr", "party_code", "bioname"]],
    on="icpsr"
)


fig = px.scatter(
    df_senate,
    title="Senate",
    x="PC1",
    y="PC2",
    color="party_code",
    hover_name="bioname"
)

fig.show()